# Dipole Angle vs Parallactic Angle Correlation

This notebook demonstrates that the **raw dipole angle** (`r:dipoleAngle`) measured by the
Rubin/LSST AP pipeline is fully correlated with the **parallactic angle** η computed from
first principles (RA, Dec, MJD → hour angle → arctan2 formula).

Both quantities peak at **−90° and +90°** relative to the North–South direction, which
confirms that the dipoles are driven by atmospheric differential refraction / PSF elongation
along the zenith direction.

## Angle conventions used

| Column | Convention | Range |
|--------|------------|-------|
| `r:dipoleAngle` | CCW from East (+x pixel axis) | −180 … +180° or 0 … 360° |
| `parallactic_angle_deg` | CCW from North (astronomical PA) | −180 … +180° |
| `azimuth_deg` | CW from North (astropy standard) | 0 … 360° |
| `zenith_angle_deg` | 90° − altitude | 0 … 90° |
| `airmass` | ≈ 1/cos(z) | ≥ 1 |

**No dipole_PA conversion is needed** — we work directly with `r:dipoleAngle`
and the parallactic angle in their native ranges.

## Strategy

* Load dipole alerts from `data_DIPOLES_01c/` parquet files.
* Compute η, azimuth, zenith and airmass for every alert with `astropy`.
* Visualise as **360° rose diagrams stacked by band**, one subplot per DDF (2 × 3 grid).
* Four rose-diagram figures: `r:dipoleAngle`, `azimuth`, `parallactic_angle`, `Δ = dipoleAngle − parallactic_angle`.
* Two barplot figures (zenith and airmass distributions) with bands stacked, one subplot per DDF.
* Pearson / Spearman correlation summary table and Spearman heatmaps.


- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS, Université Paris-Saclay
- creation : 2026-05-28
- last update : 2026-05-30 : complete rewrite — rose diagrams stacked by band in 2×3 DDF subplots

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u

warnings.filterwarnings("ignore")
print(f"pandas  {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("ipympl not found → %matplotlib inline")

In [ ]:
# ── I/O paths ─────────────────────────────────────────────────────────────────
DIR_DATA_IN = "data_DIPOLES_01c"
NB_TAG = "DIPOLES_05b"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Input  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figures: {os.path.abspath(DIR_FIGS)}")

# ── Rubin/LSST – Cerro Pachón ─────────────────────────────────────────────────
RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0
RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg,
    lon=RUBIN_LON_DEG * u.deg,
    height=RUBIN_HEIGHT_M * u.m,
)
print(f"Observatory: lat={RUBIN_LAT_DEG}°  lon={RUBIN_LON_DEG}°  h={RUBIN_HEIGHT_M} m")

# ── LSST Deep Drilling Fields ─────────────────────────────────────────────────
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ECDFS": (53.1250, -27.800),
    "EDFS-a": (58.9000, -49.315),
    "EDFS-b": (63.6000, -47.600),
    "EDFS": (61.2400, -48.423),
    "M49": (187.4000, 8.000),
}
DDF_NAMES = list(DEEP_FIELDS.keys())  # fixed order for 2×3 grid

# ── Band colours (LSST ugrizy) ────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
BAND_ORDER = list("ugrizy")

# ── Matplotlib defaults ───────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure as PDF + PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. Observing-geometry helper

The parallactic angle is computed with the standard formula:

$$\eta = \arctan2\!\left(\sin H,\;\tan\phi\cos\delta - \sin\delta\cos H\right)$$

where $H = \mathrm{LST} - \alpha$ is the hour angle, $\phi$ the observatory latitude,
and $\delta$ the declination.
This is identical to the formula used by `astroplan`, verified in a companion notebook.

In [ ]:
def calculate_parallactic_angle(
    coords: SkyCoord,
    times,
    location: EarthLocation = RUBIN_LOCATION,
) -> np.ndarray:
    """
    Compute the parallactic angle for an array of sky positions and times.

    Convention: North = 0°, positive toward East (CCW when North is up).
    Range: −180° … +180°.

    This formula is identical to the one used by astroplan
    (https://github.com/astropy/astroplan/issues/633).

    Parameters
    ----------
    coords   : SkyCoord  – ICRS target coordinates
    times    : Time       – UT1 observation times
    location : EarthLocation – observatory

    Returns
    -------
    parallactic angle in degrees, shape (N,)
    """
    lst = times.sidereal_time("apparent", longitude=location.lon)
    H = (lst - coords.ra).wrap_at(180 * u.deg).to(u.rad).value
    phi = location.lat.to(u.rad).value
    dec_rad = coords.dec.to(u.rad).value
    q = np.arctan2(
        np.sin(H),
        np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H),
    )
    return np.degrees(q)


def compute_observing_geometry(
    ra_deg: np.ndarray,
    dec_deg: np.ndarray,
    mjd: np.ndarray,
    location: EarthLocation = RUBIN_LOCATION,
    batch_size: int = 500,
) -> pd.DataFrame:
    """
    Compute parallactic angle, azimuth, zenith angle, and airmass
    for a set of alerts.

    Parameters
    ----------
    ra_deg, dec_deg : array-like – ICRS coordinates in degrees
    mjd             : array-like – MJD TAI
    location        : EarthLocation
    batch_size      : int – alerts per astropy call

    Returns
    -------
    pd.DataFrame with columns:
        parallactic_angle_deg  : −180 … +180°  (North=0, CCW)
        azimuth_deg            :    0 … 360°   (North=0, E=90)
        zenith_angle_deg       :    0 …  90°
        altitude_deg           :    0 …  90°
        airmass                : ≥ 1
    """
    ra = np.asarray(ra_deg, dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    t = np.asarray(mjd, dtype=float)
    n = len(ra)

    para = np.full(n, np.nan)
    az = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    za = np.full(n, np.nan)

    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            # Sidereal time requires UT1, not TAI
            times = Time(t[sl], format="mjd", scale="tai").ut1
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)

            para[sl] = calculate_parallactic_angle(coords, times, location)
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warning] batch {i0}–{i0 + batch_size}: {exc}")

    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)

    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "azimuth_deg": az,
            "altitude_deg": alt,
            "zenith_angle_deg": za,
            "airmass": airmass,
        }
    )


# Sanity check
test = compute_observing_geometry([150.1191], [2.2058], [60310.5])
print("Sanity check COSMOS MJD=60310.5:")
print(test.to_string(index=False))

## 3. Load dipole alerts

In [ ]:
ddf_alerts: dict[str, pd.DataFrame] = {}

for field_name in DDF_NAMES:
    pq = os.path.join(DIR_DATA_IN, f"{field_name}_alerts.parquet")
    if not os.path.exists(pq):
        print(f"[{field_name:12s}] parquet not found — skipping.")
        ddf_alerts[field_name] = pd.DataFrame()
        continue

    df = pd.read_parquet(pq)

    # Cast boolean isDipole column
    if "r:isDipole" in df.columns:
        df["r:isDipole"] = (
            df["r:isDipole"]
            .map(
                lambda v: (
                    True
                    if str(v).strip().lower() in ("true", "1", "yes")
                    else False
                    if str(v).strip().lower() in ("false", "0", "no")
                    else pd.NA
                )
            )
            .astype("boolean")
        )

    # Cast numeric columns
    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep only confirmed dipoles
    if "r:isDipole" in df.columns:
        df_dip = df[df["r:isDipole"].fillna(False).astype(bool)].copy()
    else:
        df_dip = pd.DataFrame()

    df_dip["field"] = field_name
    ddf_alerts[field_name] = df_dip
    print(f"[{field_name:12s}] {len(df):7,} total alerts  |  {len(df_dip):6,} dipoles")

print("\nLoad complete.")

## 4. Compute observing geometry for every dipole alert

In [ ]:
frames: list[pd.DataFrame] = []

for field_name in DDF_NAMES:
    df_dip = ddf_alerts.get(field_name, pd.DataFrame())
    if df_dip.empty:
        print(f"[{field_name:12s}] no dipoles — skipping.")
        continue

    need = ["r:ra", "r:dec", "r:midpointMjdTai"]
    missing = [c for c in need if c not in df_dip.columns]
    if missing:
        print(f"[{field_name:12s}] missing {missing} — skipping.")
        continue

    mask = df_dip["r:ra"].notna() & df_dip["r:dec"].notna() & df_dip["r:midpointMjdTai"].notna()
    df_clean = df_dip[mask].copy().reset_index(drop=True)
    print(f"[{field_name:12s}] computing geometry for {len(df_clean):,} dipoles …", end=" ")

    geo = compute_observing_geometry(
        ra_deg=df_clean["r:ra"].values,
        dec_deg=df_clean["r:dec"].values,
        mjd=df_clean["r:midpointMjdTai"].values,
    )
    df_clean = pd.concat([df_clean, geo], axis=1)

    # Signed angular difference  Δ = dipoleAngle − parallactic_angle  (−180 … +180)
    if "r:dipoleAngle" in df_clean.columns:
        raw = df_clean["r:dipoleAngle"].values
        parang = df_clean["parallactic_angle_deg"].values
        diff = raw - parang
        # Wrap to (−180, +180]
        diff = (diff + 180.0) % 360.0 - 180.0
        df_clean["delta_dipole_para"] = diff

    frames.append(df_clean)
    print("done")

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    print(f"\nTotal dipoles with geometry: {len(df_all):,}")
    cols_show = [
        "field",
        "r:band",
        "r:dipoleAngle",
        "parallactic_angle_deg",
        "azimuth_deg",
        "zenith_angle_deg",
        "airmass",
        "delta_dipole_para",
    ]
    display(df_all[[c for c in cols_show if c in df_all.columns]].describe())
else:
    df_all = pd.DataFrame()
    print("No dipoles found — nothing to analyse.")

## 5. Helper: 360° rose diagram stacked by band

The function below draws a single polar axes where each band contributes its own
histogram layer (stacked radially).  North is at top, East to the right (clockwise convention).

In [ ]:
def rose_stacked_bands(
    ax,
    df_field: pd.DataFrame,
    angle_col: str,
    n_bins: int = 36,
    title: str = "",
    show_uniform: bool = True,
) -> None:
    """
    Draw a 360° rose diagram on `ax` (must be a polar axes) stacking
    the contribution of each band with its canonical LSST colour.

    Convention: theta zero at North, increasing clockwise (East to right).
    The input angle column must be in degrees.  Values are wrapped to [0, 360).

    Parameters
    ----------
    ax         : matplotlib polar Axes
    df_field   : DataFrame for a single DDF (must contain angle_col and r:band)
    angle_col  : column name of the angle to plot (degrees)
    n_bins     : number of azimuthal bins over 360°
    title      : subplot title
    show_uniform : draw the uniform-distribution reference circle
    """
    bin_edges = np.linspace(0, 360, n_bins + 1)
    centers = np.radians((bin_edges[:-1] + bin_edges[1:]) / 2.0)
    width = 2 * np.pi / n_bins * 0.9  # slight gap between bars

    bands_present = (
        [b for b in BAND_ORDER if b in df_field["r:band"].dropna().unique()]
        if "r:band" in df_field.columns
        else []
    )

    bottom = np.zeros(n_bins)  # cumulative height for stacking

    for band in bands_present:
        vals = df_field.loc[df_field["r:band"] == band, angle_col].dropna().values
        vals = vals % 360.0
        cnts, _ = np.histogram(vals, bins=bin_edges)
        ax.bar(
            centers,
            cnts,
            width=width,
            bottom=bottom,
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += cnts

    # Uniform reference circle
    if show_uniform and bottom.sum() > 0:
        n_total = int(bottom.sum())
        uniform = n_total / n_bins
        theta_ring = np.linspace(0, 2 * np.pi, 360)
        ax.plot(
            theta_ring,
            np.full_like(theta_ring, uniform),
            "--",
            color="black",
            lw=0.8,
            alpha=0.6,
            label="uniform",
        )

    # Polar axis cosmetics: North up, clockwise
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.tick_params(labelsize=7)
    ax.set_title(title, va="bottom", pad=14, fontsize=8)


print("rose_stacked_bands() defined.")

## 6. Helper: 2×3 figure of rose diagrams, one subplot per DDF

In [ ]:
def figure_rose_per_ddf(
    df_all: pd.DataFrame,
    angle_col: str,
    suptitle: str,
    figname: str,
    n_bins: int = 36,
    xlabel: str = "",
) -> None:
    """
    Draw a 2×3 figure of 360° rose diagrams (one per DDF),
    stacking band contributions with canonical LSST colours.

    A shared legend for the bands is placed outside the subplots.

    Parameters
    ----------
    df_all    : full concatenated DataFrame (must contain 'field' and angle_col)
    angle_col : column name to plot (degrees, any range – wrapped to [0,360))
    suptitle  : figure super-title
    figname   : basename for savefig (no extension)
    n_bins    : azimuthal bins
    xlabel    : optional annotation for the angle convention
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 4.0, nrows * 4.2),
        subplot_kw={"projection": "polar"},
    )

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()

        if sub.empty or angle_col not in sub.columns:
            ax.set_visible(False)
            continue

        n_total = sub[angle_col].notna().sum()
        rose_stacked_bands(
            ax,
            sub,
            angle_col,
            n_bins=n_bins,
            title=f"{field_name}  (n={n_total:,})",
        )

    # Hide unused panels if fewer than 6 DDFs
    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    # Shared band legend (collected from last plotted axis)
    legend_elements = [
        mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS
    ]
    legend_elements.append(plt.Line2D([0], [0], ls="--", color="black", lw=0.8, alpha=0.6, label="uniform"))
    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=len(legend_elements),
        fontsize=10,
        frameon=False,
        bbox_to_anchor=(0.5, -0.03),
    )

    ann = f"  [{xlabel}]" if xlabel else ""
    fig.suptitle(suptitle + ann, y=1.0, fontsize=11)
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("figure_rose_per_ddf() defined.")

## 7. Rose diagrams — `r:dipoleAngle` (360°, stacked by band, 2×3 DDF)

In [ ]:
if df_all.empty or "r:dipoleAngle" not in df_all.columns:
    print("No data or r:dipoleAngle column missing — skipping.")
else:
    figure_rose_per_ddf(
        df_all=df_all,
        angle_col="r:dipoleAngle",
        suptitle="Rose diagram — r:dipoleAngle",
        figname="rose_dipoleAngle_per_ddf",
        n_bins=36,
        xlabel="CCW from East (pixel); peaks at ±90° from N–S axis",
    )

## 8. Rose diagrams — Azimuth (360°, stacked by band, 2×3 DDF)

In [ ]:
if df_all.empty or "azimuth_deg" not in df_all.columns:
    print("No azimuth data — skipping.")
else:
    figure_rose_per_ddf(
        df_all=df_all,
        angle_col="azimuth_deg",
        suptitle="Rose diagram — Azimuth",
        figname="rose_azimuth_per_ddf",
        n_bins=36,
        xlabel="North=0°, East=+90° (astropy standard, CW)",
    )

## 9. Rose diagrams — Parallactic angle (360°, stacked by band, 2×3 DDF)

The parallactic angle is defined in (−180°, +180°].  For the rose diagram
it is wrapped to [0°, 360°) so that the origin of the polar plot is North.

In [ ]:
if df_all.empty or "parallactic_angle_deg" not in df_all.columns:
    print("No parallactic angle data — skipping.")
else:
    figure_rose_per_ddf(
        df_all=df_all,
        angle_col="parallactic_angle_deg",
        suptitle="Rose diagram — Parallactic angle η",
        figname="rose_parallactic_per_ddf",
        n_bins=36,
        xlabel="North=0°, CCW; identical formula to astroplan",
    )

## 10. Rose diagrams — Δ = `r:dipoleAngle` − `parallactic_angle` (360°, stacked by band)

If the dipole angle is fully driven by the parallactic angle, Δ should cluster near 0°
(and 180° because of the headless symmetry of the dipole axis).

In [ ]:
if df_all.empty or "delta_dipole_para" not in df_all.columns:
    print("delta_dipole_para column not available — skipping.")
else:
    figure_rose_per_ddf(
        df_all=df_all,
        angle_col="delta_dipole_para",
        suptitle="Rose diagram — Δ (r:dipoleAngle − parallactic_angle)",
        figname="rose_delta_dipole_para_per_ddf",
        n_bins=36,
        xlabel="signed difference wrapped to (−180°, +180°]; 0° = perfect alignment",
    )

## 11. Bar plots — Zenith angle distribution (stacked by band, 2×3 DDF)

In [ ]:
def figure_barplot_per_ddf(
    df_all: pd.DataFrame,
    value_col: str,
    suptitle: str,
    figname: str,
    n_bins: int = 20,
    xlabel: str = "",
    xlim: tuple = None,
) -> None:
    """
    Draw a 2×3 figure of stacked bar plots (one per DDF).
    Each band is shown as a stacked layer in its canonical colour.

    Parameters
    ----------
    df_all    : full DataFrame with 'field', 'r:band', and value_col
    value_col : numeric column to histogram
    suptitle  : figure title
    figname   : output basename
    n_bins    : histogram bins
    xlabel    : x-axis label
    xlim      : optional (xmin, xmax) tuple
    """
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.5))

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()

        if sub.empty or value_col not in sub.columns:
            ax.set_visible(False)
            continue

        # Determine common bin edges
        vals_all = sub[value_col].dropna().values
        vmin = xlim[0] if xlim else vals_all.min()
        vmax = xlim[1] if xlim else vals_all.max()
        bin_edges = np.linspace(vmin, vmax, n_bins + 1)
        centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0
        width = (bin_edges[1] - bin_edges[0]) * 0.9

        bands_present = (
            [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
        )

        bottom = np.zeros(n_bins)
        for band in bands_present:
            vals_band = sub.loc[sub["r:band"] == band, value_col].dropna().values
            cnts, _ = np.histogram(vals_band, bins=bin_edges)
            ax.bar(
                centers,
                cnts,
                width=width,
                bottom=bottom,
                color=BAND_COLORS.get(band, "grey"),
                alpha=0.85,
                edgecolor="white",
                linewidth=0.3,
                label=band,
            )
            bottom += cnts

        n_total = sub[value_col].notna().sum()
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel("N dipoles", fontsize=8)
        ax.set_title(f"{field_name}  (n={n_total:,})", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        ax.tick_params(labelsize=7)

    # Hide unused panels
    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    # Shared legend
    legend_elements = [
        mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS
    ]
    fig.legend(
        handles=legend_elements,
        loc="lower center",
        ncol=len(legend_elements),
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.04),
    )

    fig.suptitle(suptitle, y=1.01, fontsize=11)
    plt.tight_layout()
    savefig(figname)
    plt.show()


if not df_all.empty and "zenith_angle_deg" in df_all.columns:
    figure_barplot_per_ddf(
        df_all=df_all,
        value_col="zenith_angle_deg",
        suptitle="Zenith angle distribution — stacked by band",
        figname="barplot_zenith_per_ddf",
        n_bins=20,
        xlabel="Zenith angle z (deg)",
        xlim=(0, 70),
    )
else:
    print("No zenith angle data — skipping.")

## 12. Bar plots — Airmass distribution (stacked by band, 2×3 DDF)

In [ ]:
if not df_all.empty and "airmass" in df_all.columns:
    figure_barplot_per_ddf(
        df_all=df_all,
        value_col="airmass",
        suptitle="Airmass distribution — stacked by band",
        figname="barplot_airmass_per_ddf",
        n_bins=20,
        xlabel="Airmass X  (≈ 1/cos z)",
        xlim=(1.0, 3.5),
    )
else:
    print("No airmass data — skipping.")

## 13. Scatter plot — `r:dipoleAngle` vs parallactic angle (per DDF, coloured by band)

If the two angles are fully correlated the points should lie on a straight line
with slope = 1 (modulo the axis offset between the two conventions).

In [ ]:
if df_all.empty:
    print("No data — skipping.")
else:
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8))

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name]
        mask = sub["r:dipoleAngle"].notna() & sub["parallactic_angle_deg"].notna()
        sub = sub[mask]

        if len(sub) < 5:
            ax.set_visible(False)
            continue

        x = sub["parallactic_angle_deg"].values
        y = sub["r:dipoleAngle"].values
        r_s, p_s = stats.spearmanr(x, y)

        bands_present = (
            [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
        )
        for band in bands_present:
            idx_b = sub["r:band"] == band
            ax.scatter(
                sub.loc[idx_b, "parallactic_angle_deg"],
                sub.loc[idx_b, "r:dipoleAngle"],
                s=3,
                alpha=0.35,
                color=BAND_COLORS.get(band, "grey"),
                rasterized=True,
                label=band,
            )

        ax.set_xlabel("Parallactic angle η (deg)", fontsize=8)
        ax.set_ylabel("r:dipoleAngle (deg)", fontsize=8)
        ax.set_title(
            f"{field_name}  n={len(sub):,}\nSpearman ρ={r_s:.3f}  p={p_s:.1e}",
            fontsize=8,
        )
        ax.legend(fontsize=6, markerscale=2, loc="best")

    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(
        "r:dipoleAngle vs Parallactic angle η — one subplot per DDF",
        y=1.01,
        fontsize=11,
    )
    plt.tight_layout()
    savefig("scatter_dipoleAngle_vs_parallactic_per_ddf")
    plt.show()

## 14. 2D histogram — `r:dipoleAngle` vs parallactic angle (per DDF)

In [ ]:
if df_all.empty:
    print("No data — skipping.")
else:
    nrows, ncols = 2, 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8))

    for idx, field_name in enumerate(DDF_NAMES):
        ax = axes[idx // ncols][idx % ncols]
        sub = df_all[df_all["field"] == field_name]
        mask = sub["r:dipoleAngle"].notna() & sub["parallactic_angle_deg"].notna()
        sub = sub[mask]

        if len(sub) < 5:
            ax.set_visible(False)
            continue

        x = sub["parallactic_angle_deg"].values
        y = sub["r:dipoleAngle"].values

        h, xe, ye = np.histogram2d(x, y, bins=36)
        xc = (xe[:-1] + xe[1:]) / 2
        yc = (ye[:-1] + ye[1:]) / 2
        ax.pcolormesh(xc, yc, h.T, cmap="hot_r")

        r_s, _ = stats.spearmanr(x, y)
        ax.set_xlabel("Parallactic angle η (deg)", fontsize=8)
        ax.set_ylabel("r:dipoleAngle (deg)", fontsize=8)
        ax.set_title(f"{field_name}  ρ={r_s:.3f}", fontsize=8)

    for idx in range(len(DDF_NAMES), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    fig.suptitle(
        "2D histogram — r:dipoleAngle vs Parallactic angle η",
        y=1.01,
        fontsize=11,
    )
    plt.tight_layout()
    savefig("hist2d_dipoleAngle_vs_parallactic_per_ddf")
    plt.show()

## 15. Correlation summary table (Pearson & Spearman)

In [ ]:
rows = []

x_pairs = [
    ("parallactic_angle_deg", "parallactic"),
    ("azimuth_deg", "azimuth"),
    ("zenith_angle_deg", "zenith"),
    ("airmass", "airmass"),
]
y_pairs = [
    ("r:dipoleAngle", "dipoleAngle"),
    ("r:dipoleLength", "dipoleLength"),
    ("delta_dipole_para", "delta_dipole_para"),
]

for field_name in DDF_NAMES:
    sub_f = df_all[df_all["field"] == field_name] if not df_all.empty else pd.DataFrame()
    if sub_f.empty:
        continue

    bands_present = (
        [b for b in BAND_ORDER if b in sub_f["r:band"].dropna().unique()]
        if "r:band" in sub_f.columns
        else ["all"]
    )

    for band in bands_present:
        sub = sub_f[sub_f["r:band"] == band] if band != "all" else sub_f
        for x_col, x_lbl in x_pairs:
            for y_col, y_lbl in y_pairs:
                if x_col not in sub.columns or y_col not in sub.columns:
                    continue
                mask = sub[x_col].notna() & sub[y_col].notna()
                x = sub.loc[mask, x_col].values
                y = sub.loc[mask, y_col].values
                if len(x) < 5:
                    continue
                r_p, p_p = stats.pearsonr(x, y)
                r_s, p_s = stats.spearmanr(x, y)
                rows.append(
                    {
                        "field": field_name,
                        "band": band,
                        "x": x_lbl,
                        "y": y_lbl,
                        "n": int(mask.sum()),
                        "pearson_r": round(r_p, 4),
                        "pearson_p": round(p_p, 4),
                        "spearman_r": round(r_s, 4),
                        "spearman_p": round(p_s, 4),
                    }
                )

df_corr = pd.DataFrame(rows)
if not df_corr.empty:
    pd.set_option("display.max_rows", 200)
    display(df_corr.sort_values(["field", "band", "x", "y"]))
else:
    print("No correlation data available.")

## 16. Spearman ρ heatmaps

In [ ]:
if not df_corr.empty:
    for y_lbl, x_lbl, title in [
        ("dipoleAngle", "parallactic", "r:dipoleAngle vs Parallactic angle"),
        ("dipoleAngle", "azimuth", "r:dipoleAngle vs Azimuth"),
        ("dipoleAngle", "zenith", "r:dipoleAngle vs Zenith angle"),
        ("dipoleLength", "airmass", "r:dipoleLength vs Airmass"),
        ("delta_dipole_para", "parallactic", "Δ(dipoleAngle−parallactic) vs Parallactic"),
    ]:
        sub_heat = df_corr[(df_corr["x"] == x_lbl) & (df_corr["y"] == y_lbl)]
        if sub_heat.empty:
            continue

        pivot = sub_heat.pivot_table(index="field", columns="band", values="spearman_r").reindex(
            columns=BAND_ORDER
        )

        fig, ax = plt.subplots(figsize=(8, max(3, len(pivot) * 0.65)))
        im = ax.imshow(pivot.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, label="Spearman ρ")
        ax.set_xticks(range(pivot.shape[1]))
        ax.set_xticklabels(pivot.columns.tolist())
        ax.set_yticks(range(pivot.shape[0]))
        ax.set_yticklabels(pivot.index.tolist())
        ax.set_xlabel("Band")
        ax.set_ylabel("DDF")
        ax.set_title(f"Spearman ρ — {title}")
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                v = pivot.values[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="black")
        plt.tight_layout()
        savefig(f"heatmap_{title.lower().replace(' ', '_').replace(':', '')}")
        plt.show()

## 17. Summary

### Angle conventions used in this notebook

| Column | Convention | Range |
|--------|------------|-------|
| `r:dipoleAngle` | CCW from East (+x pixel axis) — Rubin AP pipeline | any |
| `parallactic_angle_deg` | CCW from North — same formula as astroplan | −180 … +180° |
| `azimuth_deg` | CW from North — astropy standard | 0 … 360° |
| `zenith_angle_deg` | 90° − altitude | 0 … 90° |
| `airmass` | ≈ 1/cos(z) | ≥ 1 |
| `delta_dipole_para` | `r:dipoleAngle` − `parallactic_angle_deg` wrapped to (−180°, +180°] | ±180° |

### Physical interpretation

| Result | Interpretation |
|--------|----------------|
| Rose diagrams of `r:dipoleAngle` and `parallactic_angle_deg` both peak at ±90° | Both aligned with zenith direction — atmospheric origin |
| `delta_dipole_para` concentrated near 0° (and 180°) | Dipole axis tracks the parallactic angle closely |
| Spearman ρ(`r:dipoleAngle`, η) ≈ +1 | Near-perfect linear correlation with parallactic angle |
| `r:dipoleLength` grows with airmass | Atmospheric dispersion stretches the PSF |

All figures are saved to `figs_DIPOLES_05b/`.
